In [6]:
%%sql
-- Chay 1 lan: chuan bi brz_watermark cho flow NB_POI_POSTGRES_SYNC_PREPARE
-- (dong bo incremental poi_master/poi_detail -> stg_poi_master_sync/stg_poi_detail_sync)

ALTER TABLE lh_vv_bronze.dbo.brz_watermark ADD COLUMN source_table_name STRING;

INSERT INTO lh_vv_bronze.dbo.brz_watermark (
    watermark_id, table_name, watermark_column, watermark_value, update_time,
    status, flow_name, watermark_sequence, last_run_id, error_message, source_table_name
)
VALUES
    ('stg_poi_master_by_partner_sync', 'stg_poi_master_sync', 'updated_at',
     TIMESTAMP'2026-09-17 08:00:00.000000', TIMESTAMP'2026-09-17 08:00:00.000000',
     'active', 'NB_POI_POSTGRES_SYNC_PREPARE', NULL, NULL, NULL, 'poi_master'),
    ('stg_poi_detail_by_partner_sync', 'stg_poi_detail_sync', 'updated_at',
     TIMESTAMP'2026-09-17 08:00:00.000000', TIMESTAMP'2026-09-17 08:00:00.000000',
     'active', 'NB_POI_POSTGRES_SYNC_PREPARE', NULL, NULL, NULL, 'poi_detail');

ALTER TABLE lh_vv_bronze.dbo.brz_watermark ADD COLUMN filter_expression STRING;

UPDATE lh_vv_bronze.dbo.brz_watermark
SET filter_expression = "ingestion_source = 'partner_portal'"
WHERE watermark_id IN ('stg_poi_master_by_partner_sync', 'stg_poi_detail_by_partner_sync');

StatementMeta(, 544fd4ea-8b44-4e9e-9c85-357f315d4cda, 12, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
DELETE FROM lh_vv_gold.dbo.stg_poi_master_sync
WHERE producer_run_id IS NULL;

DELETE FROM lh_vv_gold.dbo.stg_poi_detail_sync
WHERE producer_run_id IS NULL;

UPDATE lh_vv_bronze.dbo.brz_watermark
SET watermark_value = TIMESTAMP'2026-09-16 08:00:00.000000', update_time = current_timestamp(), error_message = NULL
WHERE watermark_id = 'stg_poi_master_by_partner_sync';

UPDATE lh_vv_bronze.dbo.brz_watermark
SET watermark_value = TIMESTAMP'2026-09-16 08:00:00.000000', update_time = current_timestamp(), error_message = NULL
WHERE watermark_id = 'stg_poi_detail_by_partner_sync';

StatementMeta(, 90b3e248-6d5a-4b4e-8086-d6ece3e57db1, 7, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
# =============================================================================
# NOTEBOOK: nb_poi_postgres_sync_prepare.py
# MUC DICH: Doc incremental tu Gold source tables (poi_master, poi_detail) theo
#           watermark, ghi append vao staging tables (stg_poi_master_sync,
#           stg_poi_detail_sync) de chuan bi day sang Postgres.
#
# Watermark dieu khien flow nay duoc luu trong brz_watermark voi
# flow_name = 'NB_POI_POSTGRES_SYNC_PREPARE' va status = 'active'. Moi row gom:
#   - source_table_name : bang Gold doc incremental tu (vd. poi_master)
#   - table_name         : bang staging duoc ghi append vao (vd. stg_poi_master_sync)
#   - watermark_column   : cot dung lam cursor (vd. updated_at)
#   - watermark_value    : moc duoi (exclusive) cua lan doc nay
#
# Notebook khong hardcode ten bang — them 1 row watermark moi la co them 1 cap
# source/staging duoc dong bo, khong can sua code.
# =============================================================================

import json, uuid
from datetime import datetime, timezone
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

PIPELINE_NAME = "NB_POI_POSTGRES_SYNC_PREPARE"
RUN_ID        = str(uuid.uuid4())

WATERMARK_TABLE    = "lh_vv_bronze.dbo.brz_watermark"
PIPELINE_RUN_TABLE = "lh_vv_bronze.dbo.brz_pipeline_run"
GOLD_SCHEMA        = "lh_vv_gold.dbo"
CURRENT_NOTEBOOK   = "nb_poi_postgres_sync_prepare"

print(f"PIPELINE : {PIPELINE_NAME}")
print(f"RUN_ID   : {RUN_ID}")

watermark_rows = (
    spark.table(WATERMARK_TABLE)
    .filter(
        (F.col("flow_name") == PIPELINE_NAME) &
        (F.col("status") == "active")
    )
    .select(
        "watermark_id", "table_name", "source_table_name",
        "watermark_column", "watermark_value", "filter_expression",
    )
    .collect()
)

if not watermark_rows:
    print(f"Debug '{PIPELINE_NAME}'.")
else:
    print(f"Debug {len(watermark_rows)} watermark row:")
    for r in watermark_rows:
        print(f"  - {r['watermark_id']}: {r['source_table_name']} -> {r['table_name']} "
              f"(cursor={r['watermark_column']}, tu {r['watermark_value']}, "
              f"filter={r['filter_expression'] or '1=1'})")

StatementMeta(, 90b3e248-6d5a-4b4e-8086-d6ece3e57db1, 8, Finished, Available, Finished, False)

PIPELINE : NB_POI_POSTGRES_SYNC_PREPARE
RUN_ID   : aeeca129-11e1-4ad9-8b8e-480e50678929
Debug 2 watermark row:
  - stg_poi_master_by_partner_sync: poi_master -> stg_poi_master_sync (cursor=updated_at, tu 2026-09-16 08:00:00)
  - stg_poi_detail_by_partner_sync: poi_detail -> stg_poi_detail_sync (cursor=updated_at, tu 2026-09-16 08:00:00)


In [4]:
# =============================================================================
# GHI RUN LOG (RUNNING) VAO brz_pipeline_run
# =============================================================================

pipeline_run_schema = StructType([
    StructField("run_id",           StringType(),    True),
    StructField("flow_name",        StringType(),    True),
    StructField("status",           StringType(),    True),
    StructField("watermark_from",   TimestampType(), True),
    StructField("watermark_to",     TimestampType(), True),
    StructField("raw_event_count",  LongType(),      True),
    StructField("current_notebook", StringType(),    True),
    StructField("started_at",       TimestampType(), True),
    StructField("ended_at",         TimestampType(), True),
    StructField("error_message",    StringType(),    True),
    StructField("duration",         StringType(),    True),
])

run_started_at = datetime.utcnow()

spark.createDataFrame(
    [(
        RUN_ID, PIPELINE_NAME, "RUNNING",
        None, None, None,
        CURRENT_NOTEBOOK, run_started_at, None, None, None,
    )],
    schema=pipeline_run_schema,
).write.format("delta").mode("append").saveAsTable(PIPELINE_RUN_TABLE)

print(f"brz_pipeline_run: RUNNING — RUN_ID {RUN_ID}")

StatementMeta(, 90b3e248-6d5a-4b4e-8086-d6ece3e57db1, 9, Finished, Available, Finished, False)

✓ brz_pipeline_run: RUNNING — RUN_ID aeeca129-11e1-4ad9-8b8e-480e50678929


In [5]:
# =============================================================================
# DOC INCREMENTAL TU SOURCE, APPEND VAO STAGING, ADVANCE WATERMARK
#
# Watermark moi = MAX(watermark_column) cua chinh cac row vua doc duoc trong
# batch nay (khong dung gio he thong) — tranh lech gio giua Spark cluster va
# nguon du lieu. Neu batch rong, watermark giu nguyen (khong advance) de lan
# sau van xet lai dung cua so nay.
#
# sync_watermark_row() xu ly 1 watermark row doc lap
# =============================================================================

def sync_watermark_row(r: dict, run_cutoff: datetime) -> dict:
    """Dong bo 1 watermark row: doc incremental tu source_table, append vao
    staging table (table_name), advance watermark neu co du lieu moi.
    """
    watermark_id      = r["watermark_id"]
    source_table      = f"{GOLD_SCHEMA}.{r['source_table_name']}"
    target_table      = f"{GOLD_SCHEMA}.{r['table_name']}"
    watermark_column  = r["watermark_column"]
    last_watermark    = r["watermark_value"]
    filter_expression = r["filter_expression"] if r["filter_expression"] else "1=1"

    print(f"\n--- {watermark_id}: {source_table} -> {target_table} ---")
    print(f"  Filter bo sung: {filter_expression}")

    incremental_df = None
    try:
        incremental_df = (
            spark.table(source_table)
            .filter(
                (F.col(watermark_column) > F.lit(last_watermark)) &
                (F.col(watermark_column) <= F.lit(run_cutoff)) &
                F.expr(filter_expression)
            )
            .withColumn("producer_flow", F.lit(PIPELINE_NAME))
            .withColumn("producer_run_id", F.lit(RUN_ID))
            .withColumn("staged_at", F.current_timestamp())
            .withColumn("change_token", F.expr("uuid()"))
            .persist()
        )

        stats = incremental_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.max(F.col(watermark_column)).alias("max_watermark"),
        ).first()
        row_count = stats["row_count"]
        print(f"  Rows moi ({watermark_column} > {last_watermark} den {run_cutoff}): {row_count:,}")

        if row_count > 0:
            (
                incremental_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(target_table)
            )
            print(f"  Da append {row_count:,} rows vao {target_table}")

            new_watermark = stats["max_watermark"]
            DeltaTable.forName(spark, WATERMARK_TABLE).update(
                F.col("watermark_id") == F.lit(watermark_id),
                {
                    "watermark_value": F.lit(new_watermark),
                    "update_time":     F.current_timestamp(),
                    "last_run_id":     F.lit(RUN_ID),
                    "error_message":   F.lit(None).cast("string"),
                },
            )
            print(f"  Watermark advance -> {new_watermark}")
            watermark_to = new_watermark
        else:
            DeltaTable.forName(spark, WATERMARK_TABLE).update(
                F.col("watermark_id") == F.lit(watermark_id),
                {
                    "update_time":   F.current_timestamp(),
                    "last_run_id":   F.lit(RUN_ID),
                    "error_message": F.lit(None).cast("string"),
                },
            )
            print(f"  Khong co row moi — watermark giu nguyen o {last_watermark}")
            watermark_to = last_watermark

        return {
            "watermark_id":   watermark_id,
            "success":        True,
            "row_count":      row_count,
            "watermark_from": last_watermark,
            "watermark_to":   watermark_to,
            "error_message":  None,
        }

    except Exception as e:
        error_message = str(e)[:4000]
        DeltaTable.forName(spark, WATERMARK_TABLE).update(
            F.col("watermark_id") == F.lit(watermark_id),
            {
                "update_time":   F.current_timestamp(),
                "last_run_id":   F.lit(RUN_ID),
                "error_message": F.lit(error_message),
            },
        )
        print(f"  LOI: {error_message}")
        print(f"  Watermark KHONG advance — se doc lai tu {last_watermark} o lan chay sau")

        return {
            "watermark_id":   watermark_id,
            "success":        False,
            "row_count":      0,
            "watermark_from": last_watermark,
            "watermark_to":   last_watermark,
            "error_message":  f"{watermark_id}: {error_message}",
        }

    finally:
        if incremental_df is not None:
            incremental_df.unpersist()


run_cutoff = datetime.utcnow()  # can tren cua cua so doc cho run nay

sync_results = [sync_watermark_row(r, run_cutoff) for r in watermark_rows]

total_row_count = sum(res["row_count"] for res in sync_results)
watermark_froms = [res["watermark_from"] for res in sync_results]
watermark_tos   = [res["watermark_to"] for res in sync_results]
error_messages  = [res["error_message"] for res in sync_results if res["error_message"]]
has_success     = any(res["success"] for res in sync_results)
has_failure     = any(not res["success"] for res in sync_results)

StatementMeta(, 90b3e248-6d5a-4b4e-8086-d6ece3e57db1, 10, Finished, Available, Finished, False)


--- stg_poi_master_by_partner_sync: lh_vv_gold.dbo.poi_master -> lh_vv_gold.dbo.stg_poi_master_sync ---
  Rows moi (updated_at > 2026-09-16 08:00:00 den 2026-09-17 09:46:25.123051): 36
  Da append 36 rows vao lh_vv_gold.dbo.stg_poi_master_sync
  Watermark advance -> 2026-09-16 11:45:22.780989

--- stg_poi_detail_by_partner_sync: lh_vv_gold.dbo.poi_detail -> lh_vv_gold.dbo.stg_poi_detail_sync ---
  Rows moi (updated_at > 2026-09-16 08:00:00 den 2026-09-17 09:46:25.123051): 31,062
  Da append 31,062 rows vao lh_vv_gold.dbo.stg_poi_detail_sync
  Watermark advance -> 2026-09-17 03:05:53.628707


In [6]:
# =============================================================================
# DONG RUN LOG (SUCCESS / FAILED / PARTIAL_SUCCESS) trong brz_pipeline_run
# =============================================================================

run_ended_at = datetime.utcnow()

if has_failure and has_success:
    final_status = "PARTIAL_SUCCESS"
elif has_failure:
    final_status = "FAILED"
else:
    final_status = "SUCCESS"

wm_from        = F.lit(min(watermark_froms)) if watermark_froms else F.lit(None).cast("timestamp")
wm_to          = F.lit(max(watermark_tos))   if watermark_tos   else F.lit(None).cast("timestamp")
combined_error = "; ".join(error_messages)[:4000] if error_messages else None

DeltaTable.forName(spark, PIPELINE_RUN_TABLE).update(
    F.col("run_id") == F.lit(RUN_ID),
    {
        "status":          F.lit(final_status),
        "watermark_from":  wm_from,
        "watermark_to":    wm_to,
        "raw_event_count": F.lit(total_row_count),
        "ended_at":        F.lit(run_ended_at),
        "error_message":   F.lit(combined_error),
        "duration":        F.lit(f"{(run_ended_at - run_started_at).total_seconds():.3f}s"),
    },
)

print(f"\nbrz_pipeline_run: {final_status} — RUN_ID {RUN_ID} — "
      f"{total_row_count:,} rows — {(run_ended_at - run_started_at).total_seconds():.2f}s")

if final_status in ("FAILED", "PARTIAL_SUCCESS"):
    raise RuntimeError(combined_error)

StatementMeta(, 90b3e248-6d5a-4b4e-8086-d6ece3e57db1, 11, Finished, Available, Finished, False)


✓ brz_pipeline_run: SUCCESS — RUN_ID aeeca129-11e1-4ad9-8b8e-480e50678929 — 31,098 rows — 63.67s


In [ ]:
# Fabric Notebook: NB_POI_POSTGRES_SYNC_PREPARE_V4_PROD_RECOVERY
# Production self-recovery prepare for POI PostgreSQL sync.
# Pipeline concurrency MUST be 1.

# import json, uuid
# from datetime import datetime, timezone

BRONZE_DB = "lh_vv_bronze.dbo"
GOLD_DB = "lh_vv_gold.dbo"
PIPELINE_RUN_TBL = f"{BRONZE_DB}.brz_pipeline_run"
FLOW_NAME = "POI_POSTGRES_SYNC"
NOTEBOOK_NAME = "NB_POI_POSTGRES_SYNC_PREPARE_V4_1_EXIT_SAFE"

ENTITY_CONFIG = {
    "poi_master": {"target": f"{GOLD_DB}.poi_master", "shared_staging": f"{GOLD_DB}.stg_poi_master_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_master", "sync_key": "poi_id"},
    "poi_detail": {"target": f"{GOLD_DB}.poi_detail", "shared_staging": f"{GOLD_DB}.stg_poi_detail_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_detail", "sync_key": "poi_id"},
    "poi_enrichment_display": {"target": f"{GOLD_DB}.poi_enrichment_display", "shared_staging": f"{GOLD_DB}.stg_poi_enrichment_display_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_enrichment_display", "sync_key": "poi_id"},
    "poi_review": {"target": f"{GOLD_DB}.poi_review", "shared_staging": f"{GOLD_DB}.stg_poi_review_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_review", "sync_key": "review_id"},
}
REQUIRED_SHARED_META = {"change_token"}
REQUIRED_COPY_META = {"change_token", "sync_run_id", "snapshot_at"}
RECOVERABLE_AUDIT_STATUSES = {"SNAPSHOT_READY", "RECOVERY_READY", "FINALIZING", "FINALIZE_FAILED", "FAILED"}
NEW_SYNC_RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc)


def qcol(n): return f"`{str(n).replace('`','``')}`"
def sql_string(v): return "NULL" if v is None else "'" + str(v).replace("'", "''") + "'"
def table_exists(t):
    try: spark.table(t).schema; return True
    except Exception: return False

def cols(t): return list(spark.table(t).columns)
def colset(t): return set(cols(t))
def count(t): return int(spark.sql(f"SELECT COUNT(*) c FROM {t}").first()["c"])
def scalar(sql, name="c"): return spark.sql(sql).first()[name]

def validate_required(t, required, label):
    missing = sorted(set(required) - colset(t))
    if missing: raise RuntimeError(f"{label}_SCHEMA_MISMATCH table={t} missing={missing}")

def validate_key_token(t, key, label):
    nk = int(scalar(f"SELECT COUNT(*) c FROM {t} WHERE {qcol(key)} IS NULL"))
    if nk: raise RuntimeError(f"{label}_NULL_SYNC_KEY table={t} key={key} count={nk}")
    dup = int(scalar(f"SELECT COUNT(*) c FROM (SELECT {qcol(key)} FROM {t} WHERE {qcol(key)} IS NOT NULL GROUP BY {qcol(key)} HAVING COUNT(*)>1) x"))
    if dup: raise RuntimeError(f"{label}_DUPLICATE_SYNC_KEY table={t} key={key} groups={dup}")
    bt = int(scalar(f"SELECT COUNT(*) c FROM {t} WHERE change_token IS NULL OR trim(change_token)=''"))
    if bt: raise RuntimeError(f"{label}_MISSING_CHANGE_TOKEN table={t} count={bt}")

def audit_cols(): return colset(PIPELINE_RUN_TBL)

def insert_audit(run_id):
    ac = audit_cols()
    values = {"run_id":sql_string(run_id),"flow_name":sql_string(FLOW_NAME),"status":sql_string("RUNNING"),"watermark_from":"NULL","watermark_to":"NULL","raw_event_count":"0","current_notebook":sql_string(NOTEBOOK_NAME),"started_at":"current_timestamp()","ended_at":"NULL","duration":"NULL","error_message":"NULL"}
    cs=[c for c in values if c in ac]
    spark.sql(f"INSERT INTO {PIPELINE_RUN_TBL} ({','.join(qcol(c) for c in cs)}) VALUES ({','.join(values[c] for c in cs)})")

def update_audit(run_id, status, error=None, close=False):
    ac=audit_cols(); a=[]
    if "status" in ac: a.append(f"status={sql_string(status)}")
    if "current_notebook" in ac: a.append(f"current_notebook={sql_string(NOTEBOOK_NAME)}")
    if "error_message" in ac: a.append(f"error_message={sql_string(str(error)[:4000] if error else None)}")
    if close and "ended_at" in ac: a.append("ended_at=current_timestamp()")
    if a: spark.sql(f"UPDATE {PIPELINE_RUN_TBL} SET {','.join(a)} WHERE run_id={sql_string(run_id)}")

def audit_row(run_id):
    r=spark.sql(f"SELECT * FROM {PIPELINE_RUN_TBL} WHERE run_id={sql_string(run_id)} LIMIT 1").first()
    return None if r is None else r.asDict(recursive=True)

def notebook_exit(payload):
    s=json.dumps(payload,ensure_ascii=False,default=str); print("="*100); print(s); print("="*100)
    try: notebookutils.notebook.exit(s)
    except NameError: pass

def validate_contracts():
    if not table_exists(PIPELINE_RUN_TBL): raise RuntimeError(f"PIPELINE_RUN_TABLE_NOT_FOUND table={PIPELINE_RUN_TBL}")
    validate_required(PIPELINE_RUN_TBL,{"run_id","flow_name","status","started_at"},"PIPELINE_RUN")
    for entity,cfg in ENTITY_CONFIG.items():
        for k in ("target","shared_staging","copy_staging"):
            if not table_exists(cfg[k]): raise RuntimeError(f"TABLE_NOT_FOUND entity={entity} role={k} table={cfg[k]}")
        business=cols(cfg["target"]); key=cfg["sync_key"]
        if key not in business: raise RuntimeError(f"GOLD_SYNC_KEY_MISSING entity={entity} key={key}")
        validate_required(cfg["shared_staging"],set(business)|REQUIRED_SHARED_META|{key},"SHARED_STAGING")
        validate_required(cfg["copy_staging"],set(business)|REQUIRED_COPY_META|{key},"COPY_STAGING")
        validate_key_token(cfg["shared_staging"],key,"SHARED_STAGING")
        if count(cfg["copy_staging"])>0: validate_key_token(cfg["copy_staging"],key,"COPY_STAGING")

def copy_state():
    out={}; all_runs=set()
    for entity,cfg in ENTITY_CONFIG.items():
        t=cfg["copy_staging"]; n=count(t); runs=[]
        if n:
            bad=int(scalar(f"SELECT COUNT(*) c FROM {t} WHERE sync_run_id IS NULL OR trim(sync_run_id)=''"))
            if bad: raise RuntimeError(f"COPY_STG_MISSING_SYNC_RUN_ID entity={entity} count={bad}")
            runs=[str(r["sync_run_id"]) for r in spark.sql(f"SELECT DISTINCT sync_run_id FROM {t} ORDER BY sync_run_id").collect()]
            if len(runs)!=1: raise RuntimeError(f"COPY_STG_MULTIPLE_RUN_IDS entity={entity} runs={runs}")
            all_runs.update(runs)
        out[entity]={"table":t,"sync_key":cfg["sync_key"],"row_count":n,"sync_run_ids":runs}
    if len(all_runs)>1: raise RuntimeError(f"COPY_STG_INCONSISTENT_RUN_IDS run_ids={sorted(all_runs)} state={out}")
    return out, (next(iter(all_runs)) if all_runs else None)

def shared_state():
    return {e:{"table":c["shared_staging"],"sync_key":c["sync_key"],"row_count":count(c["shared_staging"])} for e,c in ENTITY_CONFIG.items()}

def snapshot_entity(entity,cfg,run_id):
    business=cols(cfg["target"]); shared=cfg["shared_staging"]; cp=cfg["copy_staging"]
    if count(cp)!=0: raise RuntimeError(f"SNAPSHOT_TARGET_NOT_EMPTY entity={entity}")
    ins=business+["change_token","sync_run_id","snapshot_at"]
    sel=[f"s.{qcol(c)}" for c in business]+["s.change_token",sql_string(run_id),"current_timestamp()"]
    spark.sql(f"INSERT INTO {cp} ({','.join(qcol(c) for c in ins)}) SELECT {','.join(sel)} FROM {shared} s")
    n=count(cp)
    if n:
        validate_key_token(cp,cfg["sync_key"],"COPY_SNAPSHOT")
        bad=int(scalar(f"SELECT COUNT(*) c FROM {cp} WHERE sync_run_id IS NULL OR sync_run_id<>{sql_string(run_id)}"))
        if bad: raise RuntimeError(f"COPY_SNAPSHOT_RUN_ID_MISMATCH entity={entity} count={bad}")
    return n

validate_contracts()
cp_state, existing_run_id = copy_state()
sh_state = shared_state()
total_shared=sum(x["row_count"] for x in sh_state.values())
total_copy=sum(x["row_count"] for x in cp_state.values())

# Recovery has priority over current Shared state.
if existing_run_id:
    ar=audit_row(existing_run_id)
    if ar is None: raise RuntimeError(f"RECOVERY_AUDIT_RUN_NOT_FOUND sync_run_id={existing_run_id}")
    if ar.get("flow_name") not in (None,FLOW_NAME): raise RuntimeError(f"RECOVERY_FLOW_MISMATCH sync_run_id={existing_run_id} actual={ar.get('flow_name')}")
    prior_status=ar.get("status")
    if prior_status=="SUCCESS": raise RuntimeError(f"COPY_STG_PRESENT_FOR_SUCCESS_RUN sync_run_id={existing_run_id}")
    if prior_status not in RECOVERABLE_AUDIT_STATUSES: raise RuntimeError(f"COPY_STG_RUN_NOT_RECOVERABLE sync_run_id={existing_run_id} status={prior_status}")

    # If PREPARE itself failed, some entity snapshots may never have been created.
    # Complete only empty COPY STG tables. For all later states, empty means either
    # legitimate zero-row snapshot or already-finalized entity and MUST stay empty.
    completed_partial_prepare=[]
    if prior_status=="FAILED":
        for entity,cfg in ENTITY_CONFIG.items():
            if count(cfg["copy_staging"])==0 and count(cfg["shared_staging"])>0:
                n=snapshot_entity(entity,cfg,existing_run_id)
                completed_partial_prepare.append({"entity":entity,"rows":n})
        cp_state, check_run = copy_state()
        if check_run != existing_run_id: raise RuntimeError(f"RECOVERY_RUN_ID_CHANGED expected={existing_run_id} actual={check_run}")

    update_audit(existing_run_id,"RECOVERY_READY",None,False)
    result={"status":"RECOVERY_READY","sync_run_id":existing_run_id,"flow_name":FLOW_NAME,"copy_data_ready":True,"recovery_mode":True,"previous_status":prior_status,"total_copy_rows":sum(x["row_count"] for x in cp_state.values()),"shared_staging":sh_state,"copy_staging":cp_state,"completed_partial_prepare":completed_partial_prepare,"next_step":"RERUN_ALL_4_COPY_DATA_THEN_FINALIZE"}
    notebook_exit(result)

# No old snapshot: create a new audit run.
# IMPORTANT: notebook_exit(...) MUST stay outside the business try/except.
# Fabric implements notebook exit as control-flow; catching it as Exception
# would incorrectly overwrite SNAPSHOT_READY/SUCCESS with FAILED.
SYNC_RUN_ID=NEW_SYNC_RUN_ID
insert_audit(SYNC_RUN_ID)

prepare_result = None
try:
    if total_shared==0:
        update_audit(SYNC_RUN_ID,"SUCCESS",None,True)
        prepare_result={
            "status":"NO_DATA",
            "sync_run_id":SYNC_RUN_ID,
            "flow_name":FLOW_NAME,
            "copy_data_ready":False,
            "recovery_mode":False,
            "total_shared_rows":0,
            "shared_staging":sh_state,
            "copy_staging":cp_state,
            "next_step":"END_SUCCESS_NO_COPY_REQUIRED"
        }
    else:
        snap={}
        for entity,cfg in ENTITY_CONFIG.items():
            n=snapshot_entity(entity,cfg,SYNC_RUN_ID)
            snap[entity]={
                "table":cfg["copy_staging"],
                "sync_key":cfg["sync_key"],
                "row_count":n,
                "status":"SNAPSHOT_READY" if n else "EMPTY"
            }
            print(f"SNAPSHOT entity={entity} rows={n}")

        total=sum(x["row_count"] for x in snap.values())
        if total<=0:
            raise RuntimeError("SNAPSHOT_EMPTY_AFTER_NON_EMPTY_SHARED_STATE")

        update_audit(SYNC_RUN_ID,"SNAPSHOT_READY",None,False)
        prepare_result={
            "status":"SNAPSHOT_READY",
            "sync_run_id":SYNC_RUN_ID,
            "flow_name":FLOW_NAME,
            "copy_data_ready":True,
            "recovery_mode":False,
            "total_shared_rows_at_precheck":total_shared,
            "total_copy_rows":total,
            "shared_staging":sh_state,
            "copy_staging":snap,
            "next_step":"RUN_ALL_4_COPY_DATA_THEN_FINALIZE"
        }
except Exception as e:
    try:
        update_audit(SYNC_RUN_ID,"FAILED",e,False)
    except Exception as ae:
        print(f"WARNING audit update failed: {ae}")
    # Never truncate COPY STG. A retry will detect this run and recover it.
    raise

# Deliberately outside try/except.
notebook_exit(prepare_result)


In [ ]:
# # Fabric Notebook: NB_POI_POSTGRES_SYNC_PREPARE_V4_PROD_RECOVERY
# # Production self-recovery prepare for POI PostgreSQL sync.
# # Pipeline concurrency MUST be 1.

# import json, uuid
# from datetime import datetime, timezone

# BRONZE_DB = "sgr_visitvn_dev.lh_vv_bronze.dbo"
# GOLD_DB = "sgr_visitvn_dev.lh_vv_gold.dbo"
# PIPELINE_RUN_TBL = f"{BRONZE_DB}.brz_pipeline_run"
# FLOW_NAME = "POI_POSTGRES_SYNC"
# NOTEBOOK_NAME = "NB_POI_POSTGRES_SYNC_PREPARE_V4_PROD_RECOVERY"

# ENTITY_CONFIG = {
#     "poi_master": {"target": f"{GOLD_DB}.poi_master", "shared_staging": f"{GOLD_DB}.stg_poi_master_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_master", "sync_key": "poi_id"},
#     "poi_detail": {"target": f"{GOLD_DB}.poi_detail", "shared_staging": f"{GOLD_DB}.stg_poi_detail_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_detail", "sync_key": "poi_id"},
#     "poi_enrichment_display": {"target": f"{GOLD_DB}.poi_enrichment_display", "shared_staging": f"{GOLD_DB}.stg_poi_enrichment_display_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_enrichment_display", "sync_key": "poi_id"},
#     "poi_review": {"target": f"{GOLD_DB}.poi_review", "shared_staging": f"{GOLD_DB}.stg_poi_review_sync", "copy_staging": f"{GOLD_DB}.copy_stg_poi_review", "sync_key": "review_id"},
# }
# REQUIRED_SHARED_META = {"change_token"}
# REQUIRED_COPY_META = {"change_token", "sync_run_id", "snapshot_at"}
# RECOVERABLE_AUDIT_STATUSES = {"SNAPSHOT_READY", "RECOVERY_READY", "FINALIZING", "FINALIZE_FAILED", "FAILED"}
# NEW_SYNC_RUN_ID = str(uuid.uuid4())
# STARTED_AT = datetime.now(timezone.utc)


# def qcol(n): return f"`{str(n).replace('`','``')}`"
# def sql_string(v): return "NULL" if v is None else "'" + str(v).replace("'", "''") + "'"
# def table_exists(t):
#     try: spark.table(t).schema; return True
#     except Exception: return False

# def cols(t): return list(spark.table(t).columns)
# def colset(t): return set(cols(t))
# def count(t): return int(spark.sql(f"SELECT COUNT(*) c FROM {t}").first()["c"])
# def scalar(sql, name="c"): return spark.sql(sql).first()[name]

# def validate_required(t, required, label):
#     missing = sorted(set(required) - colset(t))
#     if missing: raise RuntimeError(f"{label}_SCHEMA_MISMATCH table={t} missing={missing}")

# def validate_key_token(t, key, label):
#     nk = int(scalar(f"SELECT COUNT(*) c FROM {t} WHERE {qcol(key)} IS NULL"))
#     if nk: raise RuntimeError(f"{label}_NULL_SYNC_KEY table={t} key={key} count={nk}")
#     dup = int(scalar(f"SELECT COUNT(*) c FROM (SELECT {qcol(key)} FROM {t} WHERE {qcol(key)} IS NOT NULL GROUP BY {qcol(key)} HAVING COUNT(*)>1) x"))
#     if dup: raise RuntimeError(f"{label}_DUPLICATE_SYNC_KEY table={t} key={key} groups={dup}")
#     bt = int(scalar(f"SELECT COUNT(*) c FROM {t} WHERE change_token IS NULL OR trim(change_token)=''"))
#     if bt: raise RuntimeError(f"{label}_MISSING_CHANGE_TOKEN table={t} count={bt}")

# def audit_cols(): return colset(PIPELINE_RUN_TBL)

# def insert_audit(run_id):
#     ac = audit_cols()
#     values = {"run_id":sql_string(run_id),"flow_name":sql_string(FLOW_NAME),"status":sql_string("RUNNING"),"watermark_from":"NULL","watermark_to":"NULL","raw_event_count":"0","current_notebook":sql_string(NOTEBOOK_NAME),"started_at":"current_timestamp()","ended_at":"NULL","duration":"NULL","error_message":"NULL"}
#     cs=[c for c in values if c in ac]
#     spark.sql(f"INSERT INTO {PIPELINE_RUN_TBL} ({','.join(qcol(c) for c in cs)}) VALUES ({','.join(values[c] for c in cs)})")

# def update_audit(run_id, status, error=None, close=False):
#     ac=audit_cols(); a=[]
#     if "status" in ac: a.append(f"status={sql_string(status)}")
#     if "current_notebook" in ac: a.append(f"current_notebook={sql_string(NOTEBOOK_NAME)}")
#     if "error_message" in ac: a.append(f"error_message={sql_string(str(error)[:4000] if error else None)}")
#     if close and "ended_at" in ac: a.append("ended_at=current_timestamp()")
#     if a: spark.sql(f"UPDATE {PIPELINE_RUN_TBL} SET {','.join(a)} WHERE run_id={sql_string(run_id)}")

# def audit_row(run_id):
#     r=spark.sql(f"SELECT * FROM {PIPELINE_RUN_TBL} WHERE run_id={sql_string(run_id)} LIMIT 1").first()
#     return None if r is None else r.asDict(recursive=True)

# def notebook_exit(payload):
#     s=json.dumps(payload,ensure_ascii=False,default=str); print("="*100); print(s); print("="*100)
#     try: notebookutils.notebook.exit(s)
#     except NameError: pass

# def validate_contracts():
#     if not table_exists(PIPELINE_RUN_TBL): raise RuntimeError(f"PIPELINE_RUN_TABLE_NOT_FOUND table={PIPELINE_RUN_TBL}")
#     validate_required(PIPELINE_RUN_TBL,{"run_id","flow_name","status","started_at"},"PIPELINE_RUN")
#     for entity,cfg in ENTITY_CONFIG.items():
#         for k in ("target","shared_staging","copy_staging"):
#             if not table_exists(cfg[k]): raise RuntimeError(f"TABLE_NOT_FOUND entity={entity} role={k} table={cfg[k]}")
#         business=cols(cfg["target"]); key=cfg["sync_key"]
#         if key not in business: raise RuntimeError(f"GOLD_SYNC_KEY_MISSING entity={entity} key={key}")
#         validate_required(cfg["shared_staging"],set(business)|REQUIRED_SHARED_META|{key},"SHARED_STAGING")
#         validate_required(cfg["copy_staging"],set(business)|REQUIRED_COPY_META|{key},"COPY_STAGING")
#         validate_key_token(cfg["shared_staging"],key,"SHARED_STAGING")
#         if count(cfg["copy_staging"])>0: validate_key_token(cfg["copy_staging"],key,"COPY_STAGING")

# def copy_state():
#     out={}; all_runs=set()
#     for entity,cfg in ENTITY_CONFIG.items():
#         t=cfg["copy_staging"]; n=count(t); runs=[]
#         if n:
#             bad=int(scalar(f"SELECT COUNT(*) c FROM {t} WHERE sync_run_id IS NULL OR trim(sync_run_id)=''"))
#             if bad: raise RuntimeError(f"COPY_STG_MISSING_SYNC_RUN_ID entity={entity} count={bad}")
#             runs=[str(r["sync_run_id"]) for r in spark.sql(f"SELECT DISTINCT sync_run_id FROM {t} ORDER BY sync_run_id").collect()]
#             if len(runs)!=1: raise RuntimeError(f"COPY_STG_MULTIPLE_RUN_IDS entity={entity} runs={runs}")
#             all_runs.update(runs)
#         out[entity]={"table":t,"sync_key":cfg["sync_key"],"row_count":n,"sync_run_ids":runs}
#     if len(all_runs)>1: raise RuntimeError(f"COPY_STG_INCONSISTENT_RUN_IDS run_ids={sorted(all_runs)} state={out}")
#     return out, (next(iter(all_runs)) if all_runs else None)

# def shared_state():
#     return {e:{"table":c["shared_staging"],"sync_key":c["sync_key"],"row_count":count(c["shared_staging"])} for e,c in ENTITY_CONFIG.items()}

# def snapshot_entity(entity,cfg,run_id):
#     business=cols(cfg["target"]); shared=cfg["shared_staging"]; cp=cfg["copy_staging"]
#     if count(cp)!=0: raise RuntimeError(f"SNAPSHOT_TARGET_NOT_EMPTY entity={entity}")
#     ins=business+["change_token","sync_run_id","snapshot_at"]
#     sel=[f"s.{qcol(c)}" for c in business]+["s.change_token",sql_string(run_id),"current_timestamp()"]
#     spark.sql(f"INSERT INTO {cp} ({','.join(qcol(c) for c in ins)}) SELECT {','.join(sel)} FROM {shared} s")
#     n=count(cp)
#     if n:
#         validate_key_token(cp,cfg["sync_key"],"COPY_SNAPSHOT")
#         bad=int(scalar(f"SELECT COUNT(*) c FROM {cp} WHERE sync_run_id IS NULL OR sync_run_id<>{sql_string(run_id)}"))
#         if bad: raise RuntimeError(f"COPY_SNAPSHOT_RUN_ID_MISMATCH entity={entity} count={bad}")
#     return n

# validate_contracts()
# cp_state, existing_run_id = copy_state()
# sh_state = shared_state()
# total_shared=sum(x["row_count"] for x in sh_state.values())
# total_copy=sum(x["row_count"] for x in cp_state.values())

# # Recovery has priority over current Shared state.
# if existing_run_id:
#     ar=audit_row(existing_run_id)
#     if ar is None: raise RuntimeError(f"RECOVERY_AUDIT_RUN_NOT_FOUND sync_run_id={existing_run_id}")
#     if ar.get("flow_name") not in (None,FLOW_NAME): raise RuntimeError(f"RECOVERY_FLOW_MISMATCH sync_run_id={existing_run_id} actual={ar.get('flow_name')}")
#     prior_status=ar.get("status")
#     if prior_status=="SUCCESS": raise RuntimeError(f"COPY_STG_PRESENT_FOR_SUCCESS_RUN sync_run_id={existing_run_id}")
#     if prior_status not in RECOVERABLE_AUDIT_STATUSES: raise RuntimeError(f"COPY_STG_RUN_NOT_RECOVERABLE sync_run_id={existing_run_id} status={prior_status}")

#     # If PREPARE itself failed, some entity snapshots may never have been created.
#     # Complete only empty COPY STG tables. For all later states, empty means either
#     # legitimate zero-row snapshot or already-finalized entity and MUST stay empty.
#     completed_partial_prepare=[]
#     if prior_status=="FAILED":
#         for entity,cfg in ENTITY_CONFIG.items():
#             if count(cfg["copy_staging"])==0 and count(cfg["shared_staging"])>0:
#                 n=snapshot_entity(entity,cfg,existing_run_id)
#                 completed_partial_prepare.append({"entity":entity,"rows":n})
#         cp_state, check_run = copy_state()
#         if check_run != existing_run_id: raise RuntimeError(f"RECOVERY_RUN_ID_CHANGED expected={existing_run_id} actual={check_run}")

#     update_audit(existing_run_id,"RECOVERY_READY",None,False)
#     result={"status":"RECOVERY_READY","sync_run_id":existing_run_id,"flow_name":FLOW_NAME,"copy_data_ready":True,"recovery_mode":True,"previous_status":prior_status,"total_copy_rows":sum(x["row_count"] for x in cp_state.values()),"shared_staging":sh_state,"copy_staging":cp_state,"completed_partial_prepare":completed_partial_prepare,"next_step":"RERUN_ALL_4_COPY_DATA_THEN_FINALIZE"}
#     notebook_exit(result)

# # No old snapshot: create a new audit run.
# SYNC_RUN_ID=NEW_SYNC_RUN_ID
# insert_audit(SYNC_RUN_ID)
# try:
#     if total_shared==0:
#         update_audit(SYNC_RUN_ID,"SUCCESS",None,True)
#         notebook_exit({"status":"NO_DATA","sync_run_id":SYNC_RUN_ID,"flow_name":FLOW_NAME,"copy_data_ready":False,"recovery_mode":False,"total_shared_rows":0,"shared_staging":sh_state,"copy_staging":cp_state,"next_step":"END_SUCCESS_NO_COPY_REQUIRED"})

#     snap={}
#     for entity,cfg in ENTITY_CONFIG.items():
#         n=snapshot_entity(entity,cfg,SYNC_RUN_ID)
#         snap[entity]={"table":cfg["copy_staging"],"sync_key":cfg["sync_key"],"row_count":n,"status":"SNAPSHOT_READY" if n else "EMPTY"}
#         print(f"SNAPSHOT entity={entity} rows={n}")
#     total=sum(x["row_count"] for x in snap.values())
#     if total<=0: raise RuntimeError("SNAPSHOT_EMPTY_AFTER_NON_EMPTY_SHARED_STATE")
#     update_audit(SYNC_RUN_ID,"SNAPSHOT_READY",None,False)
#     notebook_exit({"status":"SNAPSHOT_READY","sync_run_id":SYNC_RUN_ID,"flow_name":FLOW_NAME,"copy_data_ready":True,"recovery_mode":False,"total_shared_rows_at_precheck":total_shared,"total_copy_rows":total,"shared_staging":sh_state,"copy_staging":snap,"next_step":"RUN_ALL_4_COPY_DATA_THEN_FINALIZE"})
# except Exception as e:
#     try: update_audit(SYNC_RUN_ID,"FAILED",e,False)
#     except Exception as ae: print(f"WARNING audit update failed: {ae}")
#     # Never truncate COPY STG. A retry will detect this run and recover it.
#     raise
